In [1]:
from src.model import SPDMatrixLearner
from src.utils import encode_df

In [2]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import numpy as np
import plotly.express as px
from time import time

In [3]:
df = pd.read_csv("datasets/large.csv")
df = df.drop(columns="sentence")

In [4]:
device = "cuda"

In [5]:
X = encode_df(df).to(device)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import logging

logger = logging.getLogger(__name__)


class PairwiseDistanceDataset(Dataset):
    def __init__(self, X, Y, init_n_pairs=32, gamma=1.1, distance=2):
        self.X = X
        self.Y = Y
        assert len(X) == len(Y)
        self.n = len(X)
        self.max_n_pairs = self.n * (self.n + 1) // 2
        self.init_n_pairs = init_n_pairs
        self.gamma = gamma
        self.n_pairs = self.init_n_pairs
        self.distance = distance

        try:
            self.p = int(self.distance)
            self.distance = lambda x, y: (x - y).norm(dim=1, p=self.p)
        except ValueError:
            if self.distance == "cosine":
                self.distance = lambda x, y: 1 - F.cosine_similarity(x, y)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        if self.n_pairs > self.max_n_pairs:
            logger.warning(
                f"Number of pairs required ({self.n_pairs}) is greater than the number of pairs ({self.max_n_pairs}). Clipping to {self.max_n_pairs}."
            )
            self.n_pairs = self.n

        n_each = int(np.sqrt(self.n_pairs))  # TODO
        ind_1 = torch.randint(
            0, self.n, (n_each,), dtype=torch.long, pin_memory=True
        )
        ind_2 = torch.randint(
            0, self.n, (n_each,), dtype=torch.long, pin_memory=True
        )

        X_1 = self.X[ind_1]
        X_2 = self.X[ind_2]
        X_dist = (X_1 - X_2).nan_to_num(0).clip(0, 1)

        Y_1 = self.Y[ind_1]
        Y_2 = self.Y[ind_2]
        Y_dist = self.distance(Y_1, Y_2)

        self.n_pairs = int(self.n_pairs * self.gamma) + 1

        return X_dist, Y_dist

In [7]:
Y = torch.rand(len(X), 100).to(device)
dataset = PairwiseDistanceDataset(X, Y, distance=2)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=lambda x: x[0])

In [8]:
torch.set_float32_matmul_precision("high")

In [13]:
model = SPDMatrixLearner(X.shape[1])
# model = torch.compile(model)
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.1, maximize=True, weight_decay=0
)
i = 0
prev_w = model.get_W().clone()
model.train()
for X, Y in dataloader:
    t = time()
    start = time()
    optimizer.zero_grad()
    Y_pred = model(X)
    loss = model.loss(Y_pred, Y)
    loss.backward()
    grad_norm = model.compute_gradient_norm()
    optimizer.step()
    duration = time() - t
    rho = model.spearman(Y_pred, Y)
    # if i % 10 == 0:
    #     # rho = spearman(Y_pred, Y[keep])
    #     # train_duration = time() - start
    #     # start = time()
    #     # keep_test = torch.randint(0, N_test, (n,))
    #     # with torch.no_grad():
    #     #     model.eval()
    #     #     Y_pred = model(X_test[keep_test])
    #     #     test_loss = criterion(Y_pred, Y_test[keep_test])
    #     #     test_rho = spearman(Y_pred, Y_test[keep_test])
    W = model.get_W()
    print(
        f"Epoch {i} - Batch size {len(X):.2g} - Train Loss: {loss.item():.3g} - Train Spearman: {rho:.3g} - Train Duration: {duration:.2g}s - Gradient Norm: {grad_norm:.2g} - Diff norm {torch.norm(W - prev_w, p="fro"):.2g} - Orig param fro {model.W.parametrizations.weight.original.norm(p="fro"):.2g}"
    )
    prev_w = model.get_W().clone()
    i += 1
    if i > 100:
        break
print("")
model.check_spd()
# with torch.no_grad():
#     print(f"Train spearman: {spearman(model(X), Y).item():.3g}")
#     print(f"Test spearman: {spearman(model(X_test), Y_test).item():.3g}")
#     print(f"Train MSE: {nn.MSELoss()(model(X), Y).item():.3g}")
#     print(f"Test MSE: {nn.MSELoss()(model(X_test), Y_test).item():.3g}")

Epoch 0 - Batch size 45 - Train Loss: 0.0402 - Train Spearman: 0.0289 - Train Duration: 0.005s - Gradient Norm: 0.25 - Diff norm 0.5 - Orig param fro 3.4
Epoch 1 - Batch size 50 - Train Loss: -0.0928 - Train Spearman: -0.115 - Train Duration: 0.0042s - Gradient Norm: 0.27 - Diff norm 0.44 - Orig param fro 3.9
Epoch 2 - Batch size 56 - Train Loss: 0.0158 - Train Spearman: 0.0202 - Train Duration: 0.0031s - Gradient Norm: 0.15 - Diff norm 0.42 - Orig param fro 4.4
Epoch 3 - Batch size 62 - Train Loss: 0.351 - Train Spearman: 0.217 - Train Duration: 0.0023s - Gradient Norm: 0.12 - Diff norm 0.25 - Orig param fro 5
Epoch 4 - Batch size 69 - Train Loss: 0.00188 - Train Spearman: 0.298 - Train Duration: 0.0026s - Gradient Norm: 0.095 - Diff norm 0.13 - Orig param fro 5.4
Epoch 5 - Batch size 76 - Train Loss: -0.109 - Train Spearman: -0.0363 - Train Duration: 0.0028s - Gradient Norm: 0.12 - Diff norm 0.12 - Orig param fro 5.8
Epoch 6 - Batch size 84 - Train Loss: 0.0867 - Train Spearman: 0.00

Number of pairs required (3335) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is great

Epoch 42 - Batch size 2.8e+03 - Train Loss: -0.00196 - Train Spearman: 0.00432 - Train Duration: 0.0046s - Gradient Norm: 0.0053 - Diff norm 0.0078 - Orig param fro 11
Epoch 43 - Batch size 3e+03 - Train Loss: 0.0199 - Train Spearman: 0.0276 - Train Duration: 0.0053s - Gradient Norm: 0.0066 - Diff norm 0.0082 - Orig param fro 11
Epoch 44 - Batch size 3.1e+03 - Train Loss: -0.0232 - Train Spearman: -0.0134 - Train Duration: 0.0055s - Gradient Norm: 0.0046 - Diff norm 0.0086 - Orig param fro 11
Epoch 45 - Batch size 3.1e+03 - Train Loss: -0.0261 - Train Spearman: -0.000326 - Train Duration: 0.0077s - Gradient Norm: 0.0064 - Diff norm 0.0082 - Orig param fro 11
Epoch 46 - Batch size 3.1e+03 - Train Loss: 0.00724 - Train Spearman: 0.00767 - Train Duration: 0.0052s - Gradient Norm: 0.0082 - Diff norm 0.0091 - Orig param fro 11
Epoch 47 - Batch size 3.1e+03 - Train Loss: -0.000185 - Train Spearman: 0.00092 - Train Duration: 0.0055s - Gradient Norm: 0.0096 - Diff norm 0.011 - Orig param fro 1

Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is great

Epoch 67 - Batch size 3.1e+03 - Train Loss: 0.00335 - Train Spearman: 0.0143 - Train Duration: 0.0053s - Gradient Norm: 0.0087 - Diff norm 0.011 - Orig param fro 11
Epoch 68 - Batch size 3.1e+03 - Train Loss: 0.0122 - Train Spearman: -0.00512 - Train Duration: 0.0056s - Gradient Norm: 0.005 - Diff norm 0.01 - Orig param fro 11
Epoch 69 - Batch size 3.1e+03 - Train Loss: 0.00384 - Train Spearman: -0.0151 - Train Duration: 0.0056s - Gradient Norm: 0.0033 - Diff norm 0.0098 - Orig param fro 11
Epoch 70 - Batch size 3.1e+03 - Train Loss: -0.00181 - Train Spearman: 0.0019 - Train Duration: 0.0049s - Gradient Norm: 0.0086 - Diff norm 0.012 - Orig param fro 11
Epoch 71 - Batch size 3.1e+03 - Train Loss: -0.00614 - Train Spearman: -0.012 - Train Duration: 0.0054s - Gradient Norm: 0.0048 - Diff norm 0.011 - Orig param fro 11
Epoch 72 - Batch size 3.1e+03 - Train Loss: 0.000878 - Train Spearman: 0.0102 - Train Duration: 0.0052s - Gradient Norm: 0.0052 - Diff norm 0.011 - Orig param fro 11
Epoch 

Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is greater than the number of samples (3120). Clipping to 3120.
Number of pairs required (3433) is great

Epoch 88 - Batch size 3.1e+03 - Train Loss: -0.0155 - Train Spearman: -0.0151 - Train Duration: 0.0047s - Gradient Norm: 0.011 - Diff norm 0.0089 - Orig param fro 12
Epoch 89 - Batch size 3.1e+03 - Train Loss: 0.0291 - Train Spearman: 0.0079 - Train Duration: 0.0064s - Gradient Norm: 0.0036 - Diff norm 0.0083 - Orig param fro 12
Epoch 90 - Batch size 3.1e+03 - Train Loss: -0.00137 - Train Spearman: -0.0183 - Train Duration: 0.0048s - Gradient Norm: 0.0066 - Diff norm 0.0095 - Orig param fro 12
Epoch 91 - Batch size 3.1e+03 - Train Loss: 0.00809 - Train Spearman: 0.00193 - Train Duration: 0.0051s - Gradient Norm: 0.0072 - Diff norm 0.0088 - Orig param fro 12
Epoch 92 - Batch size 3.1e+03 - Train Loss: 0.0219 - Train Spearman: 0.0235 - Train Duration: 0.0071s - Gradient Norm: 0.0042 - Diff norm 0.0091 - Orig param fro 12
Epoch 93 - Batch size 3.1e+03 - Train Loss: -0.00204 - Train Spearman: 0.0213 - Train Duration: 0.0054s - Gradient Norm: 0.0069 - Diff norm 0.011 - Orig param fro 12
Epo